In [1]:
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [2]:
RS = 678723
data = pd.read_csv("../../data/processed/final_dataset.csv")

feature_cols = [
    'danceability', 'energy', 'valence', 'acousticness', 'instrumentalness',
    'liveness', 'speechiness', 'tempo', 'loudness', 'duration_sec', 'popularity'
]

X = data[feature_cols]
y = data['macro_genre']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (685567, 11)
y shape: (685567,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RS, stratify=y_train_val
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RS)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

score = model.score(X_test_scaled, y_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(f"Score: {score}")

Accuracy: 0.7569686538209082
Score: 0.7569686538209082


# Class Weighting

In [4]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RS, )
model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test_scaled)

score = model.score(X_test_scaled, y_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(f"Score: {score}")

Accuracy: 0.7320550782560498
Score: 0.7320550782560498


# SMOTE

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RS)
X_train_rs, y_train_rs = smote.fit_resample(X_train_scaled, y_train)


model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RS)
model.fit(X_train_rs, y_train_rs)
y_pred = model.predict(X_test_scaled)

score = model.score(X_test_scaled, y_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(f"Score: {score}")

Accuracy: 0.7324124451186604
Score: 0.7324124451186604
